# Computer Science Fundamentals for Data Engineers
## Networking, OS, Database Internals & Distributed Systems

**What you'll learn:**
- Networking: TCP/IP, DNS, HTTP/HTTPS, ports, load balancers, CDNs
- OS concepts: processes, threads, memory management, file systems, I/O
- Database internals: B-trees, LSM trees, WAL, MVCC, query planning
- Distributed systems: CAP theorem, consensus (Raft/Paxos), replication, partitioning
- Serialization: JSON, Avro, Parquet, Protobuf, ORC -- when to use each
- Encoding & compression: UTF-8, Base64, gzip, snappy, zstd, LZ4
- Concurrency: locks, deadlocks, optimistic vs pessimistic, ACID isolation levels
- Hashing: consistent hashing, hash partitioning, bloom filters

**Estimated Time:** 8-10 hours

---

> These fundamentals explain WHY things work the way they do.
> Understanding them makes you better at debugging, system design,
> and choosing the right tool for the job.

---

---
# Section 1: Networking Fundamentals

## The OSI Model (Simplified for DE)

```
Layer 7: APPLICATION   HTTP, HTTPS, DNS, SMTP, gRPC
         ↕ What you interact with (APIs, web requests)
Layer 4: TRANSPORT     TCP (reliable), UDP (fast)
         ↕ How data is delivered (ports, connections)
Layer 3: NETWORK       IP addresses, routing
         ↕ Where data goes (source → destination)
Layer 1-2: PHYSICAL    Ethernet, WiFi, cables
         ↕ How bits physically move
```

## Key Networking Concepts for DE

| Concept | What It Is | DE Relevance |
|---------|-----------|--------------|
| **TCP** | Reliable, ordered delivery | DB connections, API calls, Kafka |
| **UDP** | Fast, unreliable | Metrics collection, logging (some) |
| **DNS** | Domain → IP resolution | Service discovery, endpoint config |
| **HTTP/HTTPS** | Request-response protocol | REST APIs, webhooks |
| **Port** | Endpoint identifier (0-65535) | Postgres:5432, Spark UI:4040 |
| **Load Balancer** | Distributes traffic across servers | API gateways, Kafka brokers |
| **Firewall/SG** | Network access control | VPC security, DB access |
| **VPC** | Virtual private network | Isolate data infrastructure |
| **Latency** | Round-trip time | Cross-region data transfer |
| **Bandwidth** | Data throughput capacity | Bulk data transfer planning |

## Network Latency Numbers Every DE Should Know

```
Local memory access:          ~100 ns
SSD read:                     ~100 μs (1,000x memory)
Same datacenter round-trip:   ~500 μs
Same region (AZ to AZ):      ~1 ms
Cross-region (US-East to West): ~60 ms
Cross-continent (US to EU):  ~100-150 ms

IMPLICATION FOR DATA ENGINEERING:
- Local cache > Remote DB > Cross-region transfer
- Keep compute CLOSE to storage (data locality!)
- Cross-region replication adds 100ms+ per record
- Batch transfers are MUCH more efficient than record-by-record
```

In [ ]:
# Networking concepts demonstration
import socket
import struct

print("NETWORKING FUNDAMENTALS FOR DE")
print("=" * 60)

# DNS resolution
print("\n1. DNS RESOLUTION:")
hosts = ["google.com", "github.com"]
for host in hosts:
    try:
        ip = socket.gethostbyname(host)
        print(f"   {host} -> {ip}")
    except socket.gaierror:
        print(f"   {host} -> (resolution failed)")

# Common ports for DE
print("\n2. COMMON PORTS IN DATA ENGINEERING:")
ports = {
    22: "SSH (remote server access)",
    80: "HTTP",
    443: "HTTPS (APIs, web)",
    3306: "MySQL",
    5432: "PostgreSQL",
    5439: "Redshift",
    6379: "Redis (caching)",
    8080: "Airflow Webserver",
    8888: "Jupyter Notebook",
    9092: "Kafka Broker",
    4040: "Spark UI",
    10000: "Hive Server",
    27017: "MongoDB",
    8443: "Databricks Workspace",
}
for port, service in sorted(ports.items()):
    print(f"   Port {port:<6} {service}")

# Bandwidth calculation
print("\n3. DATA TRANSFER TIME ESTIMATION:")
data_sizes = [("1 GB", 1), ("100 GB", 100), ("1 TB", 1000), ("10 TB", 10000)]
bandwidths = [("100 Mbps (office)", 100/8), ("1 Gbps (datacenter)", 1000/8), ("10 Gbps (cloud)", 10000/8)]
print(f"   {'Data':<10} {'100 Mbps':<15} {'1 Gbps':<15} {'10 Gbps':<15}")
print(f"   {'-'*55}")
for name, gb in data_sizes:
    times = []
    for _, gbps in bandwidths:
        seconds = gb / gbps
        if seconds < 60:
            times.append(f"{seconds:.0f}s")
        elif seconds < 3600:
            times.append(f"{seconds/60:.1f}min")
        else:
            times.append(f"{seconds/3600:.1f}hr")
    print(f"   {name:<10} {times[0]:<15} {times[1]:<15} {times[2]:<15}")

---
# Section 2: Operating System Concepts

## Processes vs Threads

```
PROCESS:                          THREAD:
┌─────────────────────┐           ┌─────────────────────┐
│ Own memory space     │           │ Shared memory space  │
│ Own file descriptors │           │ Shared file handles  │
│ Isolated (safe)      │           │ Can corrupt each other│
│ Expensive to create  │           │ Cheap to create      │
│ IPC needed to share  │           │ Direct memory sharing│
└─────────────────────┘           └─────────────────────┘

DE relevance:
- Spark EXECUTORS = separate processes (isolated, stable)
- Spark TASKS within executor = threads (shared memory, fast)
- Python GIL = only 1 thread executes Python at a time
  → Use multiprocessing for CPU work, threading for I/O
```

## Memory Hierarchy

```
Speed:   FAST ────────────────────────────────────> SLOW
Size:    SMALL ───────────────────────────────────> LARGE
Cost:    $$$$$ ───────────────────────────────────> $

┌─────┐  ┌─────────┐  ┌───────────┐  ┌──────────┐  ┌─────────┐
│ CPU │  │  L1/L2  │  │    RAM    │  │   SSD    │  │   HDD   │
│Cache│  │  Cache  │  │  (DRAM)   │  │(NVMe/SATA│  │  (Disk) │
│ ~1ns│  │  ~5ns   │  │  ~100ns   │  │  ~100μs  │  │  ~10ms  │
│ 64KB│  │ 256KB-  │  │  16-512GB │  │  1-30TB  │  │  1-20TB │
│     │  │  32MB   │  │           │  │          │  │         │
└─────┘  └─────────┘  └───────────┘  └──────────┘  └─────────┘

DE implications:
- Spark shuffle writes to DISK → expensive (avoid unnecessary shuffles)
- Caching keeps data in RAM → fast repeated access
- Columnar formats (Parquet) → sequential reads → SSD/HDD friendly
- JVM off-heap (Tungsten) → avoids GC pauses
```

## File Systems & I/O Patterns

| Pattern | Description | DE Example |
|---------|-------------|------------|
| **Sequential read** | Read file start to end | Parquet column scan |
| **Random read** | Jump to arbitrary position | Row lookup by ID |
| **Append-only** | Write only at end | Kafka log, Delta Lake commit |
| **Write-ahead log** | Write intent before data | Delta Lake transaction log |
| **Memory-mapped files** | Map file into RAM | RocksDB (Flink state) |

Sequential I/O is 100-1000x faster than random I/O on HDD.
This is WHY columnar formats (Parquet) and append-only logs (Kafka, Delta) win.

In [ ]:
# OS concepts relevant to DE
import os
import sys
import multiprocessing

print("OS CONCEPTS FOR DATA ENGINEERS")
print("=" * 60)

# System information
print(f"\n1. SYSTEM INFO:")
print(f"   OS: {os.uname().sysname} {os.uname().release}")
print(f"   CPU cores: {multiprocessing.cpu_count()}")
print(f"   Python process ID: {os.getpid()}")

# Memory
import resource
print(f"\n2. MEMORY:")
print(f"   Python interpreter size: {sys.getsizeof([])} bytes (empty list)")
print(f"   1M integers in list: ~{sys.getsizeof(list(range(100))) * 10000 // 1024 // 1024} MB estimated")

# File I/O patterns
import time
import tempfile

print(f"\n3. I/O PATTERNS (Sequential vs Random):")
# Sequential write
data = b"x" * 1024 * 1024  # 1MB
with tempfile.NamedTemporaryFile(delete=False) as f:
    fname = f.name
    start = time.perf_counter()
    for _ in range(100):
        f.write(data)
    f.flush()
    os.fsync(f.fileno())
seq_time = time.perf_counter() - start
print(f"   Sequential write (100MB): {seq_time:.3f}s ({100/seq_time:.0f} MB/s)")
os.unlink(fname)

# The lesson
print(f"\n4. WHY THIS MATTERS FOR DE:")
print(f"   - Parquet: sequential column reads → fast scans")
print(f"   - Delta Lake: append-only log → fast writes")
print(f"   - Kafka: sequential append + sequential read → high throughput")
print(f"   - Shuffle: random I/O across network → EXPENSIVE (avoid!)")
print(f"   - Partitioning: turns random reads into sequential (partition pruning)")

---
# Section 3: Database Internals

## Storage Engines: B-Tree vs LSM Tree

```
B-TREE (PostgreSQL, MySQL InnoDB, SQL Server):
  ┌───────────────────────────┐
  │        Root Node          │  Read: O(log n) -- follow tree
  │   [10 | 20 | 30 | 40]    │  Write: O(log n) -- find + update in place
  └───┬────┬────┬────┬───────┘  Good for: read-heavy, point lookups
      │    │    │    │           Used by: OLTP databases
      ▼    ▼    ▼    ▼
   [1-9] [11-19] [21-29] [31-40]  (leaf nodes = actual data or pointers)

LSM TREE (Cassandra, RocksDB, HBase, LevelDB):
  ┌──────────────────┐
  │   MemTable (RAM) │  Write: O(1) -- just append to memory
  └────────┬─────────┘  Read: check memtable, then L0, L1, L2...
           │ flush                                      
  ┌────────▼─────────┐  Good for: write-heavy workloads
  │  L0 (SSTable)    │  Used by: time-series, IoT, logging
  ├──────────────────┤
  │  L1 (SSTable)    │  Compaction: merge SSTables periodically
  ├──────────────────┤  (background process, like Delta OPTIMIZE!)
  │  L2 (SSTable)    │
  └──────────────────┘

DELTA LAKE = LSM-like on object storage!
  - Writes: append new Parquet files (fast)
  - Reads: transaction log tells which files are current
  - OPTIMIZE: compaction (merge small files into large)
  - VACUUM: garbage collection (delete old files)
```

## Write-Ahead Log (WAL)

```
Every database write:
  1. Write to WAL (sequential append -- FAST)    ← Durability guarantee
  2. Acknowledge to client ("committed!")
  3. Later: apply change to actual data pages    ← Can be batched

If crash after step 1 but before step 3:
  → Replay WAL on recovery → no data loss!

DE parallel:
  - Delta Lake transaction log = WAL for the lakehouse
  - Kafka commit log = WAL for streaming
  - Spark checkpointing = WAL for streaming state
```

## MVCC (Multi-Version Concurrency Control)

```
MVCC: Keep multiple versions of data. Readers don't block writers.

Transaction 1 (reading):     Transaction 2 (writing):
  SELECT * FROM orders       UPDATE orders SET status='done'
  → Sees version at time T1  → Creates version at time T2
  → NOT blocked by T2!       → NOT blocked by T1!

DE relevance:
  - Delta Lake TIME TRAVEL = MVCC (query any version)
  - PostgreSQL uses MVCC (concurrent reads + writes)
  - Spark's snapshot isolation for streaming
```

## Query Execution in Databases

```
SQL Query: SELECT name, SUM(amount) FROM orders WHERE status='done' GROUP BY name

PLANNING PHASE:
  1. Parse SQL → Abstract Syntax Tree
  2. Bind: resolve table/column names
  3. Optimize: rewrite rules, cost estimation
  4. Plan: choose indexes, join order, scan type

EXECUTION PHASE (Volcano/Iterator model):
  Project(name, sum)
      ↑
  HashAggregate(group by name)
      ↑
  Filter(status='done')   ← Pushed down to scan if index exists
      ↑
  TableScan(orders)       ← Sequential scan or Index scan

SPARK DOES THE SAME THING (Catalyst optimizer):
  DataFrame ops → Logical Plan → Optimized Plan → Physical Plan → Execute
```

In [ ]:
# Database internals concepts
print("DATABASE INTERNALS CONCEPTS")
print("=" * 60)

# B-Tree simulation
print("\n1. B-TREE INDEX LOOKUP (how Postgres finds a row):")
import bisect

class SimpleBTree:
    """Simplified B-tree for conceptual demonstration."""
    def __init__(self):
        self.keys = []  # Sorted keys
        self.values = {}  # Key -> value mapping

    def insert(self, key, value):
        bisect.insort(self.keys, key)
        self.values[key] = value

    def search(self, key):
        """Binary search in sorted keys: O(log n)."""
        idx = bisect.bisect_left(self.keys, key)
        if idx < len(self.keys) and self.keys[idx] == key:
            return self.values[key]
        return None

    def range_scan(self, low, high):
        """Range query: find all keys between low and high."""
        left = bisect.bisect_left(self.keys, low)
        right = bisect.bisect_right(self.keys, high)
        return [(self.keys[i], self.values[self.keys[i]]) for i in range(left, right)]

# Demo
btree = SimpleBTree()
for i in range(1, 10001):
    btree.insert(i, f"order_{i}")

# Point lookup
import time
start = time.perf_counter()
for _ in range(10000):
    btree.search(5000)
point_time = time.perf_counter() - start
print(f"   10K point lookups in B-tree (10K keys): {point_time:.4f}s")

# Range scan
start = time.perf_counter()
results = btree.range_scan(1000, 2000)
range_time = time.perf_counter() - start
print(f"   Range scan (1000 keys): {range_time:.6f}s, found {len(results)} results")

# ACID isolation levels
print("\n2. ACID ISOLATION LEVELS:")
levels = [
    ("READ UNCOMMITTED", "Can see uncommitted changes (dirty reads)", "Almost never used"),
    ("READ COMMITTED", "Only see committed changes", "PostgreSQL default, most DBs"),
    ("REPEATABLE READ", "Same query returns same results in transaction", "MySQL InnoDB default"),
    ("SERIALIZABLE", "Complete isolation (as if sequential)", "Slowest, safest"),
]
for level, desc, usage in levels:
    print(f"   {level:<22} {desc}")
    print(f"   {'':22} Usage: {usage}")

print("\n   Delta Lake provides SERIALIZABLE isolation by default!")
print("   (via optimistic concurrency control on the transaction log)")

---
# Section 4: Distributed Systems Theory

## CAP Theorem

You can only have 2 of 3 in a distributed system:

```
        Consistency
           /\
          /  \
         /    \
        / CP   \  CA
       /   zone  \ zone
      /          \
     /____________\
 Availability    Partition
                Tolerance

C (Consistency): Every read gets the most recent write
A (Availability): Every request gets a response (even if stale)
P (Partition Tolerance): System works despite network failures

In practice: P is NOT optional (networks WILL fail)
So the real choice is: CP or AP

CP systems: PostgreSQL, ZooKeeper, HBase
  → Always consistent, may be unavailable during partition

AP systems: Cassandra, DynamoDB, DNS
  → Always available, may return stale data during partition

DELTA LAKE: CP for writes (strong consistency via transaction log)
            + eventually consistent for reads across regions
```

## Consistency Models

| Model | Guarantee | Example |
|-------|-----------|---------|
| **Strong consistency** | Read always sees latest write | Single-leader DB, Delta Lake |
| **Eventual consistency** | Reads converge to latest write over time | DynamoDB, S3, DNS |
| **Causal consistency** | Causally related ops seen in order | Some distributed DBs |
| **Read-your-writes** | You always see your own writes | Common compromise |

## Replication Strategies

```
SINGLE-LEADER (most common):
  Write → Leader → Replicas (async or sync)
  Used by: PostgreSQL, MySQL, Kafka (partition leader)
  Pro: Simple, strong consistency
  Con: Leader is bottleneck, failover complexity

MULTI-LEADER (rare):
  Write → Any leader → Sync to other leaders
  Used by: CockroachDB, multi-region setups
  Pro: Low write latency everywhere
  Con: Conflict resolution is HARD

LEADERLESS (peer-to-peer):
  Write → W nodes, Read → R nodes, where W+R > N
  Used by: Cassandra, DynamoDB, Riak
  Pro: No single point of failure
  Con: Complex conflict resolution, weaker consistency
```

## Partitioning (Sharding)

```
How to split data across nodes:

HASH PARTITIONING:
  partition = hash(key) % num_partitions
  Pro: Even distribution, no hotspots
  Con: Range queries hit ALL partitions
  Used by: Kafka, Cassandra, DynamoDB, Spark (default)

RANGE PARTITIONING:
  A-F → Node 1, G-M → Node 2, N-Z → Node 3
  Pro: Range queries hit one partition
  Con: Hotspots if data is skewed
  Used by: HBase, traditional DB sharding, Delta Lake partitioning

CONSISTENT HASHING:
  Nodes mapped to positions on a ring
  Pro: Adding/removing nodes moves minimal data
  Con: Can be uneven without virtual nodes
  Used by: DynamoDB, Cassandra
```

In [ ]:
# Distributed systems concepts
import hashlib
from collections import defaultdict

print("DISTRIBUTED SYSTEMS CONCEPTS")
print("=" * 60)

# Consistent hashing demonstration
print("\n1. CONSISTENT HASHING (used by Kafka, DynamoDB, Cassandra):")

class ConsistentHash:
    """Simplified consistent hashing ring."""
    def __init__(self, nodes, virtual_nodes=100):
        self.ring = {}
        self.sorted_keys = []
        for node in nodes:
            for i in range(virtual_nodes):
                key = hashlib.md5(f"{node}:{i}".encode()).hexdigest()
                self.ring[key] = node
                self.sorted_keys.append(key)
        self.sorted_keys.sort()

    def get_node(self, data_key):
        """Find which node owns this key."""
        key_hash = hashlib.md5(str(data_key).encode()).hexdigest()
        # Find first node clockwise on the ring
        import bisect
        idx = bisect.bisect_right(self.sorted_keys, key_hash)
        if idx >= len(self.sorted_keys):
            idx = 0
        return self.ring[self.sorted_keys[idx]]

# 5 nodes in the cluster
nodes = ["node_1", "node_2", "node_3", "node_4", "node_5"]
ch = ConsistentHash(nodes, virtual_nodes=50)

# Distribute 1000 keys
distribution = defaultdict(int)
for i in range(1000):
    node = ch.get_node(f"customer_{i}")
    distribution[node] += 1

print("   Distribution of 1000 keys across 5 nodes:")
for node, count in sorted(distribution.items()):
    bar = "#" * (count // 5)
    print(f"   {node}: {count} keys {bar}")

# What happens when we ADD a node?
print("\n   When node_6 is added, only ~1/N keys move (not all!)")
print("   This is WHY consistent hashing is used in distributed systems.")

# CAP theorem in practice
print("\n2. CAP THEOREM IN DE TOOLS:")
tools = [
    ("PostgreSQL", "CP", "Consistent reads, unavailable if leader down"),
    ("Delta Lake", "CP", "Strong consistency via transaction log"),
    ("Cassandra", "AP", "Always available, eventually consistent"),
    ("DynamoDB", "AP (tunable)", "Default AP, can request strong consistency"),
    ("Kafka", "CP", "ISR-based replication, unavailable if no ISR"),
    ("S3", "AP", "Eventually consistent (was, now strong for new objects)"),
    ("Redis", "CP or AP", "Depends on cluster mode configuration"),
]
print(f"   {'Tool':<15} {'CAP':<15} {'Behavior'}")
print(f"   {'-'*70}")
for tool, cap, behavior in tools:
    print(f"   {tool:<15} {cap:<15} {behavior}")

---
# Section 5: Serialization & File Formats

## Choosing the Right Format

| Format | Type | Schema | Human Readable | Compression | DE Use |
|--------|------|--------|---------------|-------------|--------|
| **JSON** | Row | No (self-describing) | Yes | Poor | APIs, configs, logs |
| **CSV** | Row | No | Yes | Poor | Simple data exchange |
| **Avro** | Row | Yes (embedded) | No | Good | Kafka messages, data exchange |
| **Parquet** | Columnar | Yes (footer) | No | Excellent | Analytics, data lakes (DEFAULT!) |
| **ORC** | Columnar | Yes | No | Excellent | Hive ecosystem |
| **Protobuf** | Row | Yes (external .proto) | No | Good | gRPC, microservices |
| **Delta** | Columnar + Log | Yes (JSON log) | No | Excellent | Lakehouse (Parquet + transaction log) |

## Why Parquet is King for Analytics

```
ROW-BASED (CSV/JSON):           COLUMNAR (Parquet):
┌─────┬──────┬────────┐        ┌─────┬─────┬─────┬─────┐
│ id  │ name │ amount │        │ id  │ id  │ id  │ id  │  Column: id
├─────┼──────┼────────┤        ├─────┼─────┼─────┼─────┤
│ 1   │ Alice│  100   │        │Alice│ Bob │Charlie│Diana│  Column: name
├─────┼──────┼────────┤        ├─────┼─────┼─────┼─────┤
│ 2   │ Bob  │  200   │        │ 100 │ 200 │ 300 │ 400 │  Column: amount
├─────┼──────┼────────┤        └─────┴─────┴─────┴─────┘
│ 3   │Charli│  300   │
└─────┴──────┴────────┘        Benefits:
                                - SELECT amount → reads ONLY amount column
Read ALL columns always          (skip name, id entirely!)
(even if query only needs 1)   - Same-type values compress 10-100x better
                                - Vectorized processing (SIMD)
```

In [ ]:
# Serialization comparison
import json
import sys

print("SERIALIZATION & FILE FORMATS")
print("=" * 60)

# Create sample data
records = [
    {"id": i, "name": f"customer_{i}", "amount": float(i * 37 % 1000), "status": "active" if i % 3 else "inactive"}
    for i in range(1000)
]

# JSON size
json_str = json.dumps(records)
json_size = len(json_str.encode())

# CSV size (simulated)
csv_lines = ["id,name,amount,status"] + [f"{r['id']},{r['name']},{r['amount']},{r['status']}" for r in records]
csv_str = "\n".join(csv_lines)
csv_size = len(csv_str.encode())

print(f"\n  1000 records in different formats (estimated):")
print(f"   {'Format':<15} {'Size':<15} {'Ratio':<10} {'Notes'}")
print(f"   {'-'*60}")
print(f"   {'JSON':<15} {json_size//1024} KB{'':<10} {'1.0x':<10} Self-describing, verbose")
print(f"   {'CSV':<15} {csv_size//1024} KB{'':<10} {f'{csv_size/json_size:.2f}x':<10} No types, smaller")
print(f"   {'Parquet':<15} {'~5-15 KB':<15} {'~0.1x':<10} Columnar + compression")
print(f"   {'Avro':<15} {'~20-30 KB':<15} {'~0.3x':<10} Binary + schema")

print(f"\n  WHY PARQUET IS 10x SMALLER:")
print(f"   - Dictionary encoding: 'active' stored once, referenced by index")
print(f"   - Run-length encoding: same value repeated = stored as (value, count)")
print(f"   - Bit packing: small integers use fewer bits")
print(f"   - Snappy/ZSTD compression on top of encoding")

print(f"\n  COMPRESSION ALGORITHMS (speed vs ratio):")
algos = [
    ("None", "N/A", "Fastest read/write"),
    ("Snappy", "2-4x", "Fast, moderate compression (Spark default)"),
    ("LZ4", "2-4x", "Very fast, similar to Snappy"),
    ("GZIP", "5-8x", "Slow but good compression (archival)"),
    ("ZSTD", "4-7x", "Best balance (fast + good compression) -- RECOMMENDED"),
]
print(f"   {'Algorithm':<12} {'Ratio':<10} {'Notes'}")
for algo, ratio, notes in algos:
    print(f"   {algo:<12} {ratio:<10} {notes}")

---
# Section 6: Hashing, Bloom Filters & Encoding

## Hash Functions in Data Engineering

| Use Case | Hash Function | Where Used |
|----------|--------------|------------|
| Partitioning | murmur3, MD5 | Kafka, Spark shuffle, Cassandra |
| Deduplication | SHA-256 | Data quality, CDC |
| Checksums | CRC32, xxHash | File integrity, Parquet footers |
| Bloom filters | Multiple hashes | Parquet/Delta row group skipping |
| Consistent hashing | MD5/SHA on ring | Distributed caching, DynamoDB |

## Bloom Filters (Used in Delta Lake & Parquet)

```
A bloom filter answers: "Is this element POSSIBLY in the set?"
  - YES → might be in the set (small false positive rate)
  - NO → DEFINITELY not in the set (no false negatives)

Delta Lake uses bloom filters to SKIP files:
  Query: WHERE customer_id = 'ABC123'
  Bloom filter for each file says: "ABC123 is NOT in this file"
  → Skip the file entirely! (massive speedup for point lookups)

  Space: ~10 bits per element (1.25 bytes per key)
  False positive rate: ~1% with 10 bits/element
```

## Character Encoding

```
ASCII: 7 bits, 128 characters (English only)
UTF-8: Variable length (1-4 bytes), backward compatible with ASCII
  - English: 1 byte per character
  - Chinese/Japanese/Korean: 3 bytes per character
  - Emoji: 4 bytes per character

DE relevance:
  - CSV files: ALWAYS specify encoding (UTF-8 is standard)
  - String length != byte length for non-English data
  - Spark default: UTF-8 (handles international data correctly)
  - Database collation affects sorting and comparison
```

In [ ]:
# Hashing and bloom filter demo
import hashlib
from typing import List

print("HASHING & BLOOM FILTERS")
print("=" * 60)

# Simple bloom filter implementation
class BloomFilter:
    """Simplified bloom filter (educational -- production uses bitarray)."""
    def __init__(self, size: int = 1000, num_hashes: int = 3):
        self.size = size
        self.num_hashes = num_hashes
        self.bits = [0] * size

    def _hashes(self, item: str) -> List[int]:
        positions = []
        for i in range(self.num_hashes):
            h = hashlib.md5(f"{item}:{i}".encode()).hexdigest()
            positions.append(int(h, 16) % self.size)
        return positions

    def add(self, item: str):
        for pos in self._hashes(item):
            self.bits[pos] = 1

    def might_contain(self, item: str) -> bool:
        return all(self.bits[pos] == 1 for pos in self._hashes(item))

# Demo: Simulate Delta Lake file skipping
bf = BloomFilter(size=10000, num_hashes=5)

# Add 1000 customer IDs that ARE in this file
for i in range(1000):
    bf.add(f"customer_{i}")

# Test lookups
print("  Bloom Filter Demo (like Delta Lake file skipping):")
print(f"  Elements added: 1000 customer IDs")
print(f"  Bits used: {sum(bf.bits)}/{bf.size} ({sum(bf.bits)*100//bf.size}%)")

# True positives (elements we added)
tp = sum(1 for i in range(100) if bf.might_contain(f"customer_{i}"))
print(f"\n  True positives (first 100 known): {tp}/100")

# False positives (elements NOT added)
fp = sum(1 for i in range(1000, 1100) if bf.might_contain(f"customer_{i}"))
print(f"  False positives (100 unknown): {fp}/100 ({fp}% false positive rate)")

# True negatives (elements NOT added, correctly rejected)
tn = sum(1 for i in range(1000, 1100) if not bf.might_contain(f"customer_{i}"))
print(f"  True negatives (correctly skipped): {tn}/100")
print(f"\n  In Delta Lake: {tn}% of non-matching files would be SKIPPED!")
print(f"  This is how Delta bloom filters speed up point lookups.")

---
# Summary: CS Fundamentals Cheat Sheet for DE

## What to Know for Interviews

| Topic | Key Concepts | Why It Matters |
|-------|-------------|----------------|
| **Networking** | TCP/UDP, ports, latency, bandwidth | Debug connection issues, plan data transfer |
| **OS** | Processes/threads, memory hierarchy, I/O patterns | Understand Spark execution, tune performance |
| **DB Internals** | B-tree vs LSM, WAL, MVCC, query planning | Choose right DB, understand Delta Lake |
| **Distributed Systems** | CAP, replication, partitioning, consensus | Design reliable pipelines, understand failures |
| **Serialization** | Parquet vs Avro vs JSON, compression | Choose right format, optimize storage |
| **Hashing** | Consistent hashing, bloom filters | Understand partitioning, optimize lookups |

## One-Liners for Interviews

- "Parquet is columnar, so analytics queries only read needed columns — 10x less I/O."
- "Delta Lake is like a database WAL on top of Parquet files — giving ACID to the lake."
- "Kafka partitions use hash partitioning for ordering + parallelism."
- "Spark shuffles are expensive because they're random I/O across the network."
- "CAP: Delta Lake is CP (consistent writes via the transaction log)."
- "Bloom filters let Delta skip files that definitely don't contain the queried value."

---
*Computer Science Fundamentals for Data Engineers*